In [ ]:

# Parameters supplied by Azure Data Factory

dbutils.widgets.text("source_system", "")
dbutils.widgets.text("source_object", "")
dbutils.widgets.text("bronze_path", "")
dbutils.widgets.text("silver_path", "")
dbutils.widgets.text("load_id", "")

source_system = dbutils.widgets.get("source_system")
source_object = dbutils.widgets.get("source_object")
bronze_path = dbutils.widgets.get("bronze_path")
silver_path = dbutils.widgets.get("silver_path")
load_id = dbutils.widgets.get("load_id")

print("Source system:", source_system)
print("Source object:", source_object)
print("Bronze path:", bronze_path)
print("Silver path:", silver_path)
print("Load ID:", load_id)


In [ ]:

from pyspark.sql import functions as F

# Transformation
def transform_silver(df, source_object):

    if source_object.startswith("transactions_"):
        return df.withColumn(
            "TransactionDate",
            F.to_date(F.col("DateID"), "MMddyyyy")
        )

    elif source_object == "calendar":
        return (
            df.withColumn("Year", F.year("Date"))
              .withColumn("Month", F.month("Date"))
              .withColumn("Day", F.dayofmonth("Date"))
              .withColumn("Quarter", F.quarter("Date"))
              .withColumn("DayOfWeek", F.dayofweek("Date"))
        )

    elif source_object == "channel":
        return (
            df.withColumn(
                "ChannelKey",
                F.trim(F.col("ChannelKey")).cast("int")
            )
            .withColumn("Channel", F.trim(F.col("Channel")))
        )

    elif source_object == "promotion":
        result = (
            df.withColumn(
                "PromotionKey",
                F.trim(F.col("PromotionKey")).cast("int")
            )
            .withColumn(
                "DiscountPercent",
                F.trim(F.col("DiscountPercent")).cast("decimal(5,4)")
            )
        )

        for col_name in [
            "PromotionName",
            "PromotionDescription",
            "PromotionType",
            "PromotionCategory"
        ]:
            result = result.withColumn(
                col_name, F.trim(F.col(col_name))
            )

        return result

    elif source_object == "category":
        return (
            df.withColumn(
                "ProductCategoryLabel",
                F.trim(F.col("ProductCategoryLabel"))
            )
            .withColumn(
                "ProductCategoryName",
                F.trim(F.col("ProductCategoryName"))
            )
        )

    elif source_object == "subcategory":
        return (
            df.withColumn(
                "ProductSubcategoryLabel",
                F.trim(F.col("ProductSubcategoryLabel"))
            )
            .withColumn(
                "ProductSubcategoryName",
                F.trim(F.col("ProductSubcategoryName"))
            )
        )

    elif source_object == "product":
        result = df

        text_columns = [
            "ProductName", "Manufacturer", "BrandName",
            "ClassID", "ClassName", "StyleID",
            "StyleName", "ColorID", "ColorName"
        ]

        for col_name in text_columns:
            result = result.withColumn(
                col_name, F.trim(F.col(col_name))
            )

        return result

    else:
        raise ValueError(
            f"Unsupported source object: {source_object}"
        )

print("Silver transformation dispatcher loaded.")


In [ ]:

from pyspark.sql import functions as F

# Validate Silver Transformations
def validate_silver(df, source_object):

    # Required columns for each object
    required_columns = {
        "channel": ["ChannelKey", "Channel"],
        "promotion": ["PromotionKey", "DiscountPercent"],
        "category": ["ProductCategoryKey", "ProductCategoryName"],
        "subcategory": [
            "ProductSubcategoryKey",
            "ProductCategoryKey",
            "ProductSubcategoryName"
        ],
        "product": ["ProductKey", "ProductName"],
        "calendar": ["Date"],
        "transactions_2021": [
            "TransactionDate", "ProductKey",
            "ChannelKey", "SalesQuantity", "SalesAmount"
        ],
        "transactions_2022": [
            "TransactionDate", "ProductKey",
            "ChannelKey", "SalesQuantity", "SalesAmount"
        ],
        "transactions_2023": [
            "TransactionDate", "ProductKey",
            "ChannelKey", "SalesQuantity", "SalesAmount"
        ]
    }

    if source_object not in required_columns:
        raise ValueError(f"Unsupported object: {source_object}")

    # Check missing or invalid mandatory values
    conditions = []

    for column in required_columns[source_object]:
        condition = F.col(column).isNull()

        if column in [
            "Channel", "ProductName",
            "ProductCategoryName", "ProductSubcategoryName"
        ]:
            condition = condition | (F.col(column) == "")

        conditions.append(condition)

    invalid_filter = conditions[0]

    for condition in conditions[1:]:
        invalid_filter = invalid_filter | condition

    if source_object == "promotion":
        invalid_filter = (
            invalid_filter
            | (F.col("DiscountPercent") < 0)
            | (F.col("DiscountPercent") > 1)
        )

    if source_object.startswith("transactions_"):
        invalid_filter = (
            invalid_filter
            | (F.col("SalesQuantity") < 0)
        )

    invalid_count = df.filter(invalid_filter).count()

    # Dimension business-key uniqueness
    business_keys = {
        "channel": "ChannelKey",
        "promotion": "PromotionKey",
        "category": "ProductCategoryKey",
        "subcategory": "ProductSubcategoryKey",
        "product": "ProductKey",
        "calendar": "Date"
    }

    duplicate_count = 0

    if source_object in business_keys:
        key = business_keys[source_object]

        duplicate_count = (
            df.groupBy(key)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

    # Parent relationship validation
    parent_relationships = {
        "subcategory": (
            "ProductCategoryKey", "mysql/category"
        ),
        "product": (
            "ProductSubcategoryKey", "mysql/subcategory"
        )
    }

    missing_parent_count = 0

    if source_object in parent_relationships:
        key, parent_object = parent_relationships[source_object]

        parent_path = (
            "abfss://silver@sthybridlakeprojectdev"
            f".dfs.core.windows.net/{parent_object}/"
        )

        parent_df = spark.read.format("delta").load(parent_path)

        missing_parent_count = (
            df.filter(F.col(key).isNotNull())
            .select(key)
            .distinct()
            .join(
                parent_df.select(key).distinct(),
                key,
                "left_anti"
            )
            .count()
        )

    print("Object:", source_object)
    print("Invalid records:", invalid_count)
    print("Duplicate business keys:", duplicate_count)
    print("Missing parent keys:", missing_parent_count)

    assert invalid_count == 0, "Invalid records detected"
    assert duplicate_count == 0, "Duplicate business keys detected"
    assert missing_parent_count == 0, "Missing parent keys detected"

    print("Silver data quality validation PASSED")


In [ ]:

# Validate required parameters
required_params = {
    "source_system": source_system,
    "source_object": source_object,
    "bronze_path": bronze_path,
    "silver_path": silver_path,
    "load_id": load_id
}

for name, value in required_params.items():
    if not value.strip():
        raise ValueError(f"Missing parameter: {name}")

# Read Bronze and apply source-specific transformations
bronze_df = spark.read.format("delta").load(bronze_path)
silver_df = transform_silver(bronze_df, source_object)

# First execution test: Channel-specific validation
if source_object == "channel":
    invalid_count = silver_df.filter(
        F.col("ChannelKey").isNull()
        | F.col("Channel").isNull()
        | (F.col("Channel") == "")
    ).count()

    duplicate_count = (
        silver_df.groupBy("ChannelKey")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert invalid_count == 0, "Invalid channel records"
    assert duplicate_count == 0, "Duplicate ChannelKey values"

# Apply reusable data-quality validation
validate_silver(silver_df, source_object)

# Record count reconciliation
bronze_count = bronze_df.count()
silver_count = silver_df.count()

assert bronze_count == silver_count, "Row count mismatch"

print("Source:", source_system, source_object)
print("Bronze records:", bronze_count)
print("Silver records:", silver_count)

# Controlled Silver Delta write
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

# Verify persisted output
written_count = spark.read.format("delta").load(silver_path).count()
assert written_count == bronze_count, "Persisted row count mismatch"

print("Written records:", written_count)
print("Parameterized Silver transformation PASSED")
